# Load input data

In [2]:
import sys
sys.path.append("/DATA/WGS_study/YSL/projects/latentgee/examples/code")

from latentgee_prototype import get_experiment_data
X_df3, meta_df3, batch_df3, _ = get_experiment_data(
    design_id="df3" ,
    file_path="/DATA/WGS_study/YSL/projects/Data",
    verbose=True,
)

Design ID               : df3
Design name             : prep_genus_norm
Description             : Preprocessed HIVRC, genus-level, normalized
Aggregation             : genus
Normalize               : True
Cleanset Filtering      : True
Subset studies          : None
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table           : (936, 668)
meta_data               : (936, 11)
n_batches               : 14


# Trial investigation

In [3]:
import pandas as pd
import glob

# df3 CSV 파일 찾기
df3_p1_res = pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df3/phase1/optuna_trials_2026-05-14.csv")
print(f"전체 trial 수: {len(df3_p1_res)}")

# 필터링
df_clean = df3_p1_res[
    (df3_p1_res['permanova'] > 0.01) &
    (df3_p1_res['permanova'] < 0.1) &
    # (df3_p1_res['noise_ratio'] < 0.65) &
    (df3_p1_res['n_clusters'] >= 10)
].sort_values(['noise_ratio', 'permanova'], ascending=[True, True])

print(f"의미있는 trial 수: {len(df_clean)}")
print(df_clean[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']])

전체 trial 수: 1296
의미있는 trial 수: 1
     trial_number  permanova  n_clusters  noise_ratio  cutoff
954          1569     0.0668          10     0.754274     0.1


In [4]:
import optuna

study = optuna.load_study(
    study_name="latentgee_optuna_df3",
    storage="sqlite:////DATA/WGS_study/YSL/projects/latentgee/examples/latentgee_optuna.db"
)

# 상태별 trial 수 확인
from collections import Counter
state_counts = Counter(str(t.state) for t in study.trials)
for state, count in state_counts.items():
    print(f"{state}: {count}")

TrialState.COMPLETE: 2000


In [5]:
# n_clusters 기준 완화
for min_k in [10, 8, 5, 3]:
    filtered = df3_p1_res[
        (df3_p1_res['permanova'] > 0.01) &
        (df3_p1_res['permanova'] < 0.1) &
        (df3_p1_res['noise_ratio'] < 0.65) &
        (df3_p1_res['n_clusters'] >= min_k)
    ]
    print(f"n_clusters >={min_k}: {len(filtered)}개")


# 조건 완화한 best trial 확인
df_clean2 = df3_p1_res[
        (df3_p1_res['permanova'] > 0.01) &
        (df3_p1_res['permanova'] < 0.1) &
        (df3_p1_res['noise_ratio'] < 0.65)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'], ascending=[True, True, False]).head(20)

print(f"\n완화된 필터 결과: {len(df_clean2)}개")
print(df_clean2[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']])

n_clusters >=10: 0개
n_clusters >=8: 0개
n_clusters >=5: 3개
n_clusters >=3: 72개

완화된 필터 결과: 20개
     trial_number  permanova  n_clusters  noise_ratio  cutoff
228           414   0.063142           2     0.023504   0.005
72            157   0.063166           2     0.048077   0.010
38            118   0.067404           2     0.052350   0.010
426           755   0.055757           2     0.054487   0.010
249           450   0.089148           2     0.056624   0.005
673          1174   0.093238           2     0.056624   0.030
377           682   0.076630           2     0.057692   0.005
383           688   0.077679           2     0.061966   0.100
109           214   0.069542           2     0.064103   0.010
586          1008   0.061318           2     0.066239   0.070
124           236   0.068349           2     0.070513   0.010
105           212   0.093163           2     0.071581   0.050
388           692   0.057032           2     0.074786   0.005
366           662   0.058284          

In [14]:
# 필터링
df_clean = df3_p1_res[
    (df3_p1_res['permanova'] > 0.01) &
    (df3_p1_res['permanova'] < 0.1) &
    (df3_p1_res['noise_ratio'] < 0.65) &
    (df3_p1_res['n_clusters'] >= 3)
].sort_values(['noise_ratio', 'permanova'], ascending=[True, True]).head(30)

print(f"의미있는 trial 수: {len(df_clean)}")
print(df_clean[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']])

의미있는 trial 수: 30
      trial_number  permanova  n_clusters  noise_ratio  cutoff
595           1027   0.061770           3     0.092949   0.005
247            445   0.084375           3     0.092949   0.005
67             151   0.073418           3     0.100427   0.010
545            957   0.065522           3     0.105769   0.100
291            528   0.088313           3     0.147436   0.030
471            845   0.080657           3     0.181624   0.010
694           1211   0.042512           3     0.370726   0.010
649           1130   0.040797           4     0.418803   0.030
173            338   0.074082           3     0.429487   0.050
1247          1939   0.021751           3     0.436966   0.100
1084          1713   0.023152           3     0.436966   0.100
835           1439   0.084628           4     0.493590   0.100
825           1429   0.056066           3     0.503205   0.100
517            905   0.060763           4     0.508547   0.010
605           1047   0.097471         

# Best trial selection (trial 1027)

In [15]:
print(df3_p1_res[df3_p1_res['trial_number'] == 1027].T)

                               595
cutoff                       0.005
trial_number                  1027
base_dim                       768
n_layers                         3
latent_dim                      55
activation                    relu
strategy                  constant
dropout_rate                   0.5
epochs                         150
learning_rate             0.000012
loss                      1.283626
permanova                  0.06177
n_clusters                       3
noise_ratio               0.092949
min_cluster_size                 5
min_samples_token              5.0
batch_size                      64
init               kaiming_uniform
beta_kl                   0.057566
weight_decay              0.000001
grad_clip_norm            1.110795
csm                            eom
kl_warmup_ratio           0.308682
norm                     layernorm
scheduler                     none


# Run phase 2

In [11]:
from prototype_updated_phase2 import main
main(
    experiment_name="df3",
    phase=2,
    best_trial_number=1027,
    trial_res_file_phase2="/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df3/phase1/optuna_trials_2026-05-14.csv"
)

2026-05-18 13:36:55 | 로그 저장 경로: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df3/phase2/latentgee_prototype_cutoff_df3_pid2013759_2026-05-18.log
2026-05-18 13:36:55 | LatentGEE 시작 | experiment=df3 | phase=2
2026-05-18 13:36:55 | config snapshot saved: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df3/phase2/config_used.yaml
2026-05-18 13:36:55 | python == 3.10.20
2026-05-18 13:36:55 | torch == 2.2.2
2026-05-18 13:36:55 | numpy == 1.26.4
2026-05-18 13:36:55 | scikit-learn == 1.7.2
2026-05-18 13:36:55 | optuna == 3.6.1
2026-05-18 13:36:55 | hdbscan == 0.8.41
Design ID               : df3
Design name             : prep_genus_norm
Description             : Preprocessed HIVRC, genus-level, normalized
Aggregation             : genus
Normalize               : True
Cleanset Filtering      : True
Subset studies          : None
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table         

In [16]:
# from functions import zero_filter
def zero_filter(df, meta, cutoff):
    batch="Study"
    prevalence = (df > 0).sum(axis=0) / df.shape[0]
    df = df.loc[:, prevalence > best_cutoff].copy()

    row_sums = df.sum(axis=1)
    keep_sample = row_sums > 0
    df = df.loc[keep_sample].copy()
    meta = meta.loc[keep_sample.values].reset_index(drop=True)
    
    assert len(df) == len(meta)
    assert all(df.index.astype(str) == meta["SeqID"].astype(str))
    
    return df, meta, meta[batch]

best_cutoff = 0.005
X_df3_filtered, meta_df3_filtered, batch_df3_filtered = zero_filter(X_df3, meta_df3, best_cutoff)

    

In [16]:
X_df3_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df3_filtered_cutoff0.005.csv", index=True)
meta_df3_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/meta_df3_filtered_cutoff0.005.csv", index=True)
batch_df3_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/batch_df3_filtered_cutoff0.005.csv", index=True)

In [20]:
import numpy as np

# X_corrected 로드
X_corrected_df3 = pd.read_csv(
    "/DATA/WGS_study/YSL/projects/latentgee/examples/results/df3/phase2/df3_X_corrected_trial1027_cutoff0.005_2026-05-18.csv",
    index_col=0
)

# inf, NaN 처리
X_corrected_df3_clean = X_corrected_df3.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
row_sums = X_corrected_df3_clean.sum(axis=1).replace(0, 1)
X_corrected_df3_clean = X_corrected_df3_clean.div(row_sums, axis=0)

trial=1027

X_corrected_df3_clean.to_csv(f"/DATA/WGS_study/YSL/projects/latentgee/examples/results/df3/phase2/X_corrected_df3_trial{trial}_cutoff{best_cutoff}_clean_latentgee.csv", index=True)


print(f"shape: {X_corrected_df3_clean.shape}")
print(f"NaN 수: {X_corrected_df3_clean.isna().sum().sum()}")
print(f"inf 수: {np.isinf(X_corrected_df3_clean.values).sum()}")

shape: (936, 353)
NaN 수: 0
inf 수: 0


In [21]:
from functions import evaluate_batch_correction
df3_result = evaluate_batch_correction(
    X_before     = X_df3_filtered,
    X_after      = X_corrected_df3_clean,
    batch_labels = batch_df3_filtered,
    bio_labels   = meta_df3_filtered['hivstatus'],
    renormalize  = True,
    label        = "df3 — LatentGEE"
)



  df3 — LatentGEE
                        Before   After  Change
Metric                                        
PERMANOVA R² (batch) ↓  0.1835  0.0140 -0.1695
PERMANOVA R² (bio) ↑    0.0041  0.0010 -0.0031
kBET acceptance rate ↑  0.2051  0.9925  0.7874
ASW (batch) → 0        -0.1002 -0.0222  0.0780
ASW (bio) ↑            -0.0245 -0.0023  0.0223

